In [9]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
import seaborn as sns
from operator import attrgetter
import matplotlib.colors as mcolors
import numpy as np
import calendar




In [2]:
# Dataset for Funnel Analysis

df = pd.read_csv('data.csv')


df.head()

,user_id,date,category,device,event_type,__index_level_0__
0,956977,2020-10-04 00:00:00,Office Supplies,desktop,visit,0
1,941383,2020-10-25 00:00:00,Furniture,mobile,visit,1
2,864077,2021-11-19 00:00:00,Technology,mobile,visit,2
3,689619,2022-02-05 00:00:00,Technology,desktop,visit,3
4,689619,2022-02-07 00:00:00,Technology,desktop,view,4


In [4]:
event_counts = (
    df.groupby('event_type')['event_type']
    .count()
    .reset_index(name='event_count')
    .sort_values(by='event_count', ascending=False)
    .reset_index(drop=True)
)
event_counts

,event_type,event_count
0,visit,192486
1,view,150160
2,add_to_cart,79714
3,purchase,11908


In [7]:
# We'll start by making an overall funnel, which gives an 'at a glance' view of the distribution of customers at each step. 
# We need to slightly reshape our data for this, so we'll create a list that holds the count of our users at each stage in our process.

flow_count = [
    df[df['event_type'] == 'visit'].shape[0],
    df[df['event_type'] == 'view'].shape[0],
    df[df['event_type'] == 'add_to_cart'].shape[0],
    df[df['event_type'] == 'purchase'].shape[0],
]



In [17]:
stages = ['Visits', 'view product', 'add to cart', 'purchased product']
fig = go.Figure(go.Funnel( x=flow_count, y=stages, textinfo = 'value+percent initial'))
fig.show()

In [19]:
def stage_loss(df, show_all=False):
    # Ожидается датафрейм из двух колонок: [event_type, event_count]
    values = df.values.tolist()
    
    max_drop_pct = 0.0
    worst_stage = None

    for index in range(len(values) - 1):
        start_event, start_count = values[index][0], values[index][1]
        next_event, next_count = values[index + 1][0], values[index + 1][1]
        
        stage = f"{start_event} -> {next_event}"
        
        # Разница и процент отвала (drop-off rate) на текущем шаге
        difference = start_count - next_count
        drop_pct = difference / start_count if start_count > 0 else 0.0

        if show_all:
            print(f"The difference in users from {stage} is {round(100 * drop_pct, 1)}% ({difference:,} users lost)")

        # Ищем этап с максимальной относительной потерей пользователей
        if drop_pct > max_drop_pct:
            max_drop_pct = drop_pct
            worst_stage = stage

    return worst_stage, max_drop_pct

In [20]:
# 1. Формируем таблицу этапов (если еще не создана)
stages_order = ['visit', 'view', 'add_to_cart', 'purchase']
event_counts = (
    df[df['event_type'].isin(stages_order)]
    .groupby('event_type')['event_type']
    .count()
    .reindex(stages_order)
    .reset_index(name='event_count')
)

# 2. Вызываем функцию с show_all=True для вывода всех шагов
stage, percent = stage_loss(event_counts, show_all=True)

print(f"\nНаибольший отвал: {stage} ({round(100 * percent, 1)}%)")

The difference in users from visit -> view is 22.0% (42,326 users lost)
The difference in users from view -> add_to_cart is 46.9% (70,446 users lost)
The difference in users from add_to_cart -> purchase is 85.1% (67,806 users lost)

Наибольший отвал: add_to_cart -> purchase (85.1%)
